# Choosing how many epochs to train the base model

### Imports

In [1]:
import os
import json

In [2]:
# cd ..

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from models.archs.utils import init_model
from trainer.utils import training_regimen_lr_annealing
from trainer.val import validate



### Set configs for the pretraining

In [ ]:
device = "mps" if torch.mps.is_available() else "cpu"
pretraining_config = {

    "description": "Pre-training ResNet for a select number of epochs",
    
    "device": device,
    "model_class": "ResNet",
    "data": {
        "dataset": "CIFAR10",
        "batch_size": 1024,
        "num_workers": 0,
        },

    "training": {
        "num_epochs": [15],
        "num_runs": 3,
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "batch_print_freq": 5,
        },
}

### Protocol for several runs

In [5]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/jerrymoncus/.netrc.
wandb: Currently logged in as: jjmoncus (jjmoncus706) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
import os
import json
import random
import glob
import torch
import wandb
import torch.nn as nn
import torch.optim as optim
from models.archs.utils import init_model
from data.dataloaders import cifar10_dataloaders
from trainer.utils import init_folder_if_not_exists, training_regimen_lr_annealing, validate

def run_pretraining(config):

    print("-"*75)
    print("-"*13 + "  " + f"EVALUATING # OF EPOCHS FOR TRAINING {config['model_class']}" + "  " + "-"*13)
    print("-"*75 + "\n")

    # init wandb
    wandb.init(
      project="Verifying-Unlearning-2026",
      name=f"Pretraining Experiments - {config['model_class']}",
      config=config,
      reinit=True
    )

    # Make experiment results folder if it doesnt already exist
    results_folder = init_folder_if_not_exists( f"results/pretraining/seed_{config['GRAND_SEED']}" )
    
    # Save the config for this experiment to the main results folder
    with open(os.path.join(results_folder, "experiment_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # make model checkpoints folder for this seed if it doesn't exist yet
    checkpoints_folder = init_folder_if_not_exists( f"models/model_checkpoints/pretrained/seed_{config['GRAND_SEED']}" )


    for num_epochs in config["training"]["num_epochs"]:

        epoch_results_folder = init_folder_if_not_exists( os.path.join(results_folder, f"{config['model_class']}", f"{num_epochs}_epochs") )
        epoch_checkpoints_folder = init_folder_if_not_exists( os.path.join(checkpoints_folder, f"{num_epochs}_epochs") )


        # get some data (does not change in between runs)
        train_loader, _, test_loader = cifar10_dataloaders(
            data_dir="data/CIFAR10", 
            batch_size=config["data"]["batch_size"], 
            num_workers=config["data"]["num_workers"], 
            seed=config["GRAND_SEED"], 
            class_to_replace=None, 
            percent_to_replace=None,
            val=False
            )

        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #
        # ---------------- TRAIN A BASE MODEL, FROM WHICH UNLEARNING BEGINS ----------------- #
        # ----------------------------------------------------------------------------------- #
        # ----------------------------------------------------------------------------------- #


        print("-"*55)
        print("-"*13 + "  " + f"TESTING: {num_epochs} EPOCHS" + "  " + "-"*13)
        print("-"*55 + "\n")

        for i in range(1, config["training"]["num_runs"] + 1):

            # init model, opt, criterion, and scheduler
            empty_model = init_model(model_class = config["model_class"]).to(config["device"])
            criterion = nn.CrossEntropyLoss()
            opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                opt, 
                T_max=num_epochs, 
                eta_min=1e-6
            )

            # train (wandb logging underneath, dont need to re-log training accuracy)
            base_model_path = os.path.join(epoch_checkpoints_folder, f"{config['model_class']}_{i}.pth")
            trained_model, opt, scheduler, train_acc = training_regimen_lr_annealing(
                empty_model, 
                train_loader,
                opt, 
                criterion, 
                scheduler, 
                device = config["device"], 
                num_epochs=num_epochs, 
                model_path = base_model_path,
                print_freq = config["training"]["batch_print_freq"])

            # evaluate trained model on some test
            print(f"Evaluating model trained for {num_epochs} epochs on test set...\n") 
            _, test_acc, _, _, _ = validate(
                test_loader, 
                trained_model, 
                criterion, 
                print_freq = config["training"]["batch_print_freq"],
                device = config["device"]
            )
            print(f"Train accuracy: {train_acc:.4f}\n")
            print(f"Test accuracy: {test_acc:.4f}\n")
            
            results = {
                "num_epochs": num_epochs,
                "train_acc": train_acc,
                "test_acc": test_acc
                }

            # save results
            with open(os.path.join(epoch_results_folder, f"results_{i}.json"), "w") as f:
                json.dump(results, f, indent=4)

            # Save checkpoint
            print(f"Saving base model to {base_model_path}...")
            torch.save(trained_model.state_dict(), base_model_path)
                    
    wandb.finish()

    print("-"*70)
    print("-"*19 + "  " + f"FINISHED PRETRAINING" + "  " + "-"*19)
    print("-"*70 + "\n")


### Check metrics on unlearned models

In [7]:
# MAKE A RANDOM SEED
pretraining_config["GRAND_SEED"] = 2
# DO EXP
run_pretraining(config = pretraining_config)

---------------------------------------------------------------------------
-------------  EVALUATING # OF EPOCHS FOR TRAINING ResNet  -------------
---------------------------------------------------------------------------



results/pretraining/seed_2 doesn't exist - creating it...

models/model_checkpoints/pretrained/seed_2 doesn't exist - creating it...

results/pretraining/seed_2/ResNet/15_epochs doesn't exist - creating it...

models/model_checkpoints/pretrained/seed_2/15_epochs doesn't exist - creating it...



/opt/miniconda3/envs/comp0081/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


========== DATALOADER INFO
Dataset: CIFAR-10
Train: 50000 images for training
Test: 10000 images for testing
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize


-------------------------------------------------------
-------------  TESTING: 15 EPOCHS  -------------
-------------------------------------------------------

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][4/49]	Loss 2.7950 (4.6891)	Accuracy 23.438 (16.211)	Time 9.01
Epoch: [1][9/49]	Loss 1.8784 (3.3760)	Accuracy 31.445 (22.354)	Time 13.67
Epoch: [1][14/49]	Loss 1.6781 (2.8302)	Accuracy 38.867 (26.725)	Time 12.24
Epoch: [1][19/49]	Loss 1.6095 (2.5419)	Accuracy 43.066 (29.580)	Time 14.49
Epoch: [1][24/49]	Loss 1.5924 (2.3534)	Accuracy 39.453 (31.555)	Time 15.20
Epoch: [1][29/49]	Loss 1.5087 (2.2200)	Accuracy 43.652 (33.346)	Time 15.11
Epoch: [1][34/49]	Loss 1.4697 (2.1159)	Accuracy 46.289 (35.02

RAM_GB,▆▂▄▇▆▆▇▆▃▇▂▅▅▆▇▄▅▅▄▂▁▁▄▇▅▃▄▃▁▂▇█▇▇█▇▅▆▅▅
VRAM_GB,▁▇█▇███▇▇▇▇█▇▇██████████████████████████
epoch,▁▁▂▃▃▃▄▅▅▆▇▇▇█▁▁▃▃▃▄▅▅▅▆▇▇█▁▁▂▃▃▄▅▅▅▆▇▇█
learning_rate,███▇▇▆▆▅▃▃▂▂▁▁██▇▇▆▆▅▄▃▃▂▁▁███▇▇▆▅▄▃▃▂▂▁
time,▂▁▁▁▁▂▁▁▁▁▂▁▁▁▁▂▁█▁▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▃▄▅▆▇▇▇▇▇████▃▃▅▅▅▆▆▆▆▆▆▇▇▇▇████▁▂▂▇▇███
train_loss,█▆▄▂▂▁▁▁▁▁▁▁▁▁▁▁▃▂▂▂▂▁▁▁▁▁█▃▃▂▂▂▂▂▁▁▁▁▁▁
weight_norm,█▇▆▅▄▃▃▂▂▁▁▁▁▁█▇▅▄▄▃▂▂▂▂▁▁▁█▇▆▅▄▃▂▂▂▁▁▁▁
RAM_GB,0.16321
VRAM_GB,9.53789
epoch,15


----------------------------------------------------------------------
-------------------  FINISHED PRETRAINING  -------------------
----------------------------------------------------------------------

